<a href="https://colab.research.google.com/github/Py-saqlain/Movie_Recap/blob/main/Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MOVIE RECAP ENGINE | DAY 01**

In [ ]:

# MASTER SETUP — run this every session

import torch, os, subprocess, json, shutil

# Check GPU
gpu = torch.cuda.is_available()
print(f"Device: {'GPU — ' + torch.cuda.get_device_name(0) if gpu else 'CPU'}")

!apt-get install -y ffmpeg -q
print("ffmpeg ready.")

# faiss-gpu broken on Python 3.12 — faiss-cpu works fine for our project
!pip install openai-whisper faiss-cpu sentence-transformers edge-tts -q
print("Packages installed.")

# Mount Drive
from google.colab import drive
drive.mount("/content/drive")

# Paths
VIDEO     = "/content/drive/MyDrive/MovieApp/movie.mp4"
DRIVE_DIR = "/content/drive/MyDrive/MovieApp"
CLIPS_DIR = "/content/clips"
os.makedirs(CLIPS_DIR, exist_ok=True)

# Verify
print(f"Video exists  : {os.path.exists(VIDEO)}")
print(f"Drive folder  : {os.path.exists(DRIVE_DIR)}")
print("Ready.")

**DAY 02 ( 4 cells  )**

In [ ]:
os.makedirs("/content/clips", exist_ok=True)

SCENES = [
    ("scene_01", 120,  180),
    ("scene_02", 600,  680),
    ("scene_03", 2400, 2480),
    ("scene_04", 5400, 5480),
]

clip_paths = []

for name, start, end in SCENES:
    out = f"/content/clips/{name}.mp4"
    duration = end - start

    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start),
        "-i", VIDEO,
        "-t", str(duration),
        "-c:v", "libx264",    # re-encode video — forces exact frame alignment
        "-c:a", "aac",        # re-encode audio
        "-crf", "18",         # quality level — 0 best, 51 worst, 18 = near lossless
        "-preset", "ultrafast", # fastest encoding speed
        "-avoid_negative_ts", "make_zero",  # fixes timestamp issues
        out
    ], check=True)

    size = os.path.getsize(out) / 1_000_000
    clip_paths.append(out)
    print(f"{name} → {start}s to {end}s → {size:.1f} MB ✔")

print(f"\nAll 4 clips cut. Note: re-encoding takes longer than stream copy.")

In [ ]:
concat_file = "/content/concat_list.txt"
merged      = "/content/day2_merged_fixed.mp4"

with open(concat_file, "w") as f:
    for path in clip_paths:
        f.write(f"file '{path}'\n")

subprocess.run([
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", concat_file,
    "-c:v", "libx264",      # re-encode final merge too
    "-c:a", "aac",
    "-crf", "18",
    "-preset", "ultrafast",
    merged
], check=True)

size = os.path.getsize(merged) / 1_000_000
print(f"Fixed merged video → {size:.1f} MB")

In [ ]:
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json",
     "-show_format", "-show_streams", merged],
    capture_output=True, text=True
)
info = json.loads(probe.stdout)

print("Stream check:")
for s in info["streams"]:
    print(f"  {s['codec_type'].upper()} → {s.get('duration','?')}s")

actual   = float(info["format"]["duration"])
expected = sum(end - start for _, start, end in SCENES)

print(f"\nExpected : {expected}s")
print(f"Actual   : {actual:.2f}s")
print(f"Diff     : {abs(actual - expected):.2f}s")

if abs(actual - expected) < 2:
    print("SYNC CHECK PASSED ✔")

shutil.copy(merged, f"{DRIVE_DIR}/day2_merged_fixed.mp4")
print(f"Saved → day2_merged_fixed.mp4")

**DAY 03 WHISPER**

In [ ]:
import whisper, json, time

print("Loading Whisper model...")
model = whisper.load_model("small")
# small = good accuracy + reasonable speed
# base = faster but less accurate
# medium = best accuracy but slow on CPU

print(f"Transcribing full movie ({115.6} min)...")
print("This will take 40-60 min on CPU, 5-8 min on GPU.")
print("DO NOT close Colab. Let it run.")

start_time = time.time()

result = model.transcribe(
    VIDEO,
    word_timestamps=True,
    verbose=False,          # no spam output
    condition_on_previous_text=True,
    initial_prompt="This is a movie."
)

elapsed = (time.time() - start_time) / 60
print(f"\nDone in {elapsed:.1f} minutes.")
print(f"Language detected : {result['language']}")
print(f"Total segments    : {len(result['segments'])}")

In [ ]:

# DAY 3 — CELL 2 — Filter + Save
# Works for ANY movie, nothing hardcoded
# ═══════════════════════════════════════

# Get actual duration from the result itself
all_segments = result["segments"]
if all_segments:
    actual_end = all_segments[-1]["end"]
else:
    actual_end = 0

# Calculate skip boundaries dynamically
SKIP_INTRO  = 20.0
SKIP_OUTRO  = actual_end - 60.0

print(f"Movie duration from Whisper : {actual_end:.1f}s = {actual_end/60:.1f} min")
print(f"Skipping intro before       : {SKIP_INTRO}s")
print(f"Skipping outro after        : {SKIP_OUTRO:.1f}s")

# Filter segments
segments = []
for seg in all_segments:
    if seg["start"] < SKIP_INTRO:
        continue
    if seg["end"] > SKIP_OUTRO:
        continue
    if seg.get("no_speech_prob", 0) > 0.6:
        continue
    if len(seg["text"].strip()) < 3:
        continue

    segments.append({
        "id"    : seg["id"],
        "start" : round(seg["start"], 3),
        "end"   : round(seg["end"], 3),
        "text"  : seg["text"].strip()
    })

print(f"\nTotal segments after filtering : {len(segments)}")
print(f"\nFirst 5 segments:")
for s in segments[:5]:
    print(f"  [{s['start']}s → {s['end']}s]  {s['text']}")

print(f"\nLast 5 segments:")
for s in segments[-5:]:
    print(f"  [{s['start']}s → {s['end']}s]  {s['text']}")

# Save to Drive
TRANSCRIPT_PATH = f"{DRIVE_DIR}/transcript.json"
with open(TRANSCRIPT_PATH, "w", encoding="utf-8") as f:
    json.dump(segments, f, indent=2, ensure_ascii=False)

size = os.path.getsize(TRANSCRIPT_PATH) / 1000
print(f"\nTranscript saved → {TRANSCRIPT_PATH}")
print(f"File size        : {size:.1f} KB")
print("\nDay 3 COMPLETE.")
print("This file is the brain of the engine.")
print("Never delete it. Never run Whisper on full movie again.")